# 01 · Data Drift — *Has the world my model sees changed?*

**MLOps Monitoring · Coaching Session · Module 3 (Machine Learning & GenAI)**

---

### The story
It is **1 January 2022**. Your team has just launched an HDB resale price model. It learned from every resale deal from **2017 to 2021**.

Now it is **2026**. The model has been running for almost five years and nobody has checked on it.

Your boss asks one question: **"Can we still trust it?"**

We answer in three notebooks:

| Notebook | Question | Needs real prices? |
|---|---|---|
| **01 Data drift** (this one) | Do the *flats coming in* look different from the flats the model learned from? | ❌ No — we only look at the inputs |
| 02 Concept drift & performance | Is the model still *right*? Where is it wrong? | ✅ Yes |
| 03 Retraining & rollback | When do we retrain, and how do we undo a bad release? | ✅ Yes |

> **Analogy — the school uniform shop 👕**
> A shop near a primary school stocks uniforms based on last year's sales: mostly size S and M.
> One year, a secondary school opens next door. Suddenly most customers need size L and XL.
> The shop's *stocking rule* is still the same, but the *customers changed*. That is **data drift**.
> You can spot it **before** a single sale goes wrong — just by looking at who walks in.

**How to use this notebook:** run each cell from top to bottom. Cells marked **🔧 YOUR TURN** have a blank `____` to fill in. Hints are hidden under **▸ Hint**.

## 0 · Setup — load the data and "deploy" our model

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from mlops_utils import (load_hdb, make_model, score, register_model, set_champion,
                         reset_registry, NUM, CAT, FEATURES, TARGET, DEPLOY_DATE)

pd.set_option("display.float_format", "{:,.3f}".format)

df = load_hdb()          # every HDB resale deal since Jan 2017 (data.gov.sg)
print(f"{len(df):,} resale deals from {df.month.min()} to {df.month.max()}")
df[["month", "town", "flat_type", "flat_model", "floor_area_sqm",
    "remaining_lease_years", "storey", "resale_price"]].head()

We split time into two parts:

* **Reference data** = what the model *learned from* (2017–2021). Think of it as "normal".
* **Production data** = what the model has *seen since it went live* (2022 onwards).

Every drift check in this notebook compares **production** against **reference**.

In [ ]:
reference  = df[df.date <  DEPLOY_DATE]            # 2017–2021: the model's "normal"
production = df[df.date >= DEPLOY_DATE].copy()     # 2022 onwards: life after launch

# Train and "deploy" version 1 of our model (takes ~10 seconds)
model_v1 = make_model().fit(reference[FEATURES], reference[TARGET])

reset_registry()
register_model(model_v1, "v1", notes="first release", trained_until="2021-12")
set_champion("v1")

print(f"Reference:  {len(reference):,} deals  ({reference.month.min()} to {reference.month.max()})")
print(f"Production: {len(production):,} deals  ({production.month.min()} to {production.month.max()})")
print("Model v1 saved and set as the live (champion) model ✅")

---
## Part A · Look before you measure 👀

Before any statistics, **draw the two groups on top of each other**. If the shapes match, there is probably no drift.

Below: floor area and remaining lease, *training years* vs *2025*.

In [ ]:
recent = production[production.date.dt.year == 2025]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, col in zip(axes, ["floor_area_sqm", "remaining_lease_years"]):
    ax.hist(reference[col], bins=40, density=True, alpha=0.5, label="2017–2021 (training)")
    ax.hist(recent[col],    bins=40, density=True, alpha=0.5, label="2025 (production)")
    ax.set_title(col)
    ax.legend()
plt.tight_layout()
plt.show()

**What do you notice?**
* **Floor area:** the two shapes sit almost on top of each other. Flats are the same sizes as before.
* **Remaining lease:** the 2025 shape has **shifted left**. Flats sold today have *fewer years left* on their lease.

**Why?** No data bug here — **HDB flats are ageing**. A flat built in 1990 had about 68 years of lease left in 2021, and about 64 in 2025. Every year, every flat loses one year.

Eyeballing works for 2 columns, but not for 50 columns × 12 months. We need **one number per column** that says "how much did this move?" → **PSI**.

---
## Part B · PSI — the Population Stability Index 📊

> **Analogy — the bubble tea shop 🧋**
> Split last year's customers into 10 equal age groups (10% each).
> This month, count what share of customers falls in each of the *same* age groups.
> If each group still has about 10%, nothing changed. If one group now has 30% and another 2%, the crowd has changed.
> **PSI adds up those gaps** into one number.

**Recipe**
1. Cut the **reference** data into 10 buckets with the same number of rows in each (deciles).
2. Work out what % of rows falls into each bucket **then** (reference) and **now** (production).
3. For each bucket: `(now% − then%) × ln(now% / then%)`. Add up all 10.

**How to read PSI** (common industry rule of thumb, used in banking credit models for decades):

| PSI | Meaning | Traffic light |
|---|---|---|
| below 0.10 | No real change | 🟢 stable |
| 0.10 – 0.25 | Some change — keep an eye on it | 🟡 watch |
| above 0.25 | Big change — investigate | 🔴 drifted |

### 🔧 YOUR TURN — finish the PSI formula

In [ ]:
def psi(expected, actual, bins=10):
    # 1. Bucket edges from the REFERENCE data (10 equal-sized buckets)
    edges = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    edges[0], edges[-1] = -np.inf, np.inf          # catch anything outside the old range

    # 2. Share of rows in each bucket: then vs now
    exp_pct = np.histogram(expected, edges)[0] / len(expected)
    act_pct = np.histogram(actual,   edges)[0] / len(actual)

    # 3. Avoid dividing by zero for empty buckets
    exp_pct = np.clip(exp_pct, 1e-4, None)
    act_pct = np.clip(act_pct, 1e-4, None)

    # 4. PSI = sum of (now - then) x ln(now / then)
    # 🔧 YOUR TURN: fill in the gap "now% minus then%"
    return np.sum(____ * np.log(act_pct / exp_pct))

<details><summary>▸ Hint</summary>

"Now" is `act_pct` (actual, production). "Then" is `exp_pct` (expected, reference). The blank is `(act_pct - exp_pct)`.
</details>

**Test it on made-up bubble tea customers** before trusting it on real data:

In [ ]:
rng = np.random.default_rng(0)
last_year     = rng.normal(35, 10, 5000)     # customers' ages last year
same_crowd    = rng.normal(35, 10, 5000)     # nothing changed
bit_younger   = rng.normal(32, 10, 5000)     # a few more students
school_opened = rng.normal(22, 6, 5000)      # a secondary school opened next door

for name, ages in [("Same crowd", same_crowd), ("A bit younger", bit_younger),
                   ("School opened next door", school_opened)]:
    print(f"{name:<25} PSI = {psi(last_year, ages):.3f}")

You should see roughly **0.00 → ~0.1 → well above 0.25**. If so, your formula works ✅.

### Now on real HDB data — PSI for every input column, every year

In [ ]:
def traffic_light(value):
    return "🟢" if value < 0.10 else ("🟡" if value < 0.25 else "🔴")

rows = []
for year, batch in production.groupby(production.date.dt.year):
    rows.append({"year": year, **{col: psi(reference[col], batch[col]) for col in NUM}})
psi_table = pd.DataFrame(rows).set_index("year")

print(psi_table.map(lambda v: f"{traffic_light(v)} {v:.3f}").to_string())

**What do you notice?**

* `floor_area_sqm` and `storey` stay **🟢 green** every year. Flats are the same size and height as before.
* `remaining_lease_years` goes **🟡 → 🔴**, getting worse every year. This is the ageing effect we saw in Part A.

**Is red always bad?** No. PSI only says *"this column looks different"*. A human still has to ask:
1. Is it a **bug**? (e.g. someone changed lease from years to months) → fix the pipeline.
2. Is it the **real world changing**? (flats ageing, a new town opening) → check whether the model still predicts well (notebook 02).

> 💡 **PSI on the target (the price)?** Run `psi(reference[TARGET], recent[TARGET])` below. You will see a huge number. Prices drifted far more than any input — keep that in mind for notebook 02.

In [ ]:
# 🔧 YOUR TURN: PSI of the resale price, training years vs 2025
print(f"PSI of resale_price (2017–2021 vs 2025): {psi(reference[TARGET], recent[____]):.2f}")

---
## Part C · The KS test — and the "too-sensitive smoke alarm" trap 🚨

From L02 you know hypothesis tests. The **Kolmogorov–Smirnov (KS) test** asks: *"Could these two groups come from the same distribution?"*

* It gives a **statistic D** (0 to 1) = the biggest gap between the two cumulative curves → *how big* the difference is.
* And a **p-value** → *how sure* we are that some difference exists.

> **Analogy — the smoke alarm 🔥**
> A very sensitive smoke alarm goes off when you make toast. It is *technically right* (there is smoke), but it is *not useful*.
> With **tens of thousands of rows**, the KS p-value behaves like that alarm: it rings for tiny, harmless differences.

In [ ]:
then = reference["floor_area_sqm"]
now  = production.loc[production.date.dt.year == 2022, "floor_area_sqm"]

ks_all   = stats.ks_2samp(then, now)
ks_small = stats.ks_2samp(then.sample(500, random_state=1), now.sample(500, random_state=1))

print(f"Average floor area: then {then.mean():.1f} sqm, now {now.mean():.1f} sqm  (about 1 sqm apart)\n")
print(f"KS using ALL rows ({len(then):,} vs {len(now):,}):  D = {ks_all.statistic:.3f}, p-value = {ks_all.pvalue:.1e}")
print(f"KS using 500 vs 500 rows:              D = {ks_small.statistic:.3f}, p-value = {ks_small.pvalue:.2f}")
print(f"PSI:                                   {psi(then, now):.3f}  {traffic_light(psi(then, now))}")

**What do you notice?**
* With all rows, the p-value is **tiny** ("definitely different!") — for a 1 sqm difference nobody cares about.
* With 500 rows, the *same* difference is "not significant".
* **PSI and D** barely move. They measure **how big** the change is, not just whether one exists.

✅ **Rule for production:** use the p-value as a *first filter*, but decide with an **effect size** (PSI or KS D).

### 🔧 YOUR TURN — KS on remaining lease (training years vs 2025)

In [ ]:
result = stats.____(reference["remaining_lease_years"], recent["remaining_lease_years"])
print(f"D = {result.statistic:.3f}, p-value = {result.pvalue:.1e}")

<details><summary>▸ Hint</summary>

It is the same function used above: `ks_2samp`.
</details>

Compare D here with D for floor area (about 0.03). Remaining lease has a **much bigger** D — a real shift, not a toast alarm.

---
## Part D · Categories: chi-square and "who is buying where" 🏘️

KS and PSI-with-buckets work for **numbers**. For **categories** (town, flat type, flat model) we compare the **share of each category** then vs now.

* **Chi-square test** (`scipy.stats.chi2_contingency`) → is the mix different? (Same big-data smoke alarm problem as KS!)
* **Categorical PSI** → *how much* did the mix change? Same formula, but each category is its own "bucket".

> **Analogy — the hawker centre 🍜** Last year 30% of orders were chicken rice, 20% laksa…
> This year chicken rice is 18% and a new mala stall takes 15%. The *menu mix* changed.

In [ ]:
def category_shares(then, now):
    table = pd.DataFrame({"then_%": then.value_counts(normalize=True) * 100,
                          "now_%":  now.value_counts(normalize=True) * 100}).fillna(0)
    table["change_pts"] = table["now_%"] - table["then_%"]
    return table.sort_values("change_pts")

town_mix = category_shares(reference["town"], recent["town"])
print("Towns whose share of resale deals changed the most (2017–2021 vs 2025):")
pd.concat([town_mix.head(4), town_mix.tail(4)])

In [ ]:
# Chi-square test on the counts (then vs now)
counts = pd.DataFrame({"then": reference["town"].value_counts(),
                       "now":  recent["town"].value_counts()}).fillna(0)
chi2, p_value, dof, _ = stats.chi2_contingency(counts.T)
print(f"Chi-square p-value = {p_value:.1e}   <- the smoke alarm rings again")


def psi_categorical(then, now):
    then_pct = then.value_counts(normalize=True)
    now_pct  = now.value_counts(normalize=True)
    all_cats = then_pct.index.union(now_pct.index)
    then_pct = then_pct.reindex(all_cats, fill_value=0).clip(lower=1e-4)
    now_pct  = now_pct.reindex(all_cats, fill_value=0).clip(lower=1e-4)
    return float(((now_pct - then_pct) * np.log(now_pct / then_pct)).sum())


for col in CAT:
    value = psi_categorical(reference[col], recent[col])
    print(f"PSI {col:<11} = {value:.3f} {traffic_light(value)}")

**What do you notice?** The p-value says "different!", but the PSI values are all 🟢. A few towns gained or lost one or two percentage points (for example Sembawang and Bukit Batok gained share, likely as newer flats there finished their 5-year Minimum Occupation Period and could be resold), but **the overall mix of towns, flat types and models is stable**.

Keep this in mind for notebook 02: **inputs look normal… yet something will still go badly wrong.**

---
## Part E · Brand-new categories 🆕

The most dangerous drift is a value the model has **never seen**. In the HDB world:

| Real case | When will it show up in *resale* data? |
|---|---|
| **`3Gen` flat model** (3-generation flats) | ✅ Already here — first resale in **June 2022**, right *after* our model launched |
| **Tengah** (newest town) | After its first flats finish their 5-year MOP — not in the data yet |
| **Standard / Plus / Prime** classification (BTOs launched from Oct 2024) | Plus & Prime have a 10-year MOP, so roughly the **2030s** |

> **Analogy — the GPS from 2021 🗺️** Ask it to go to a road built in 2024. It does not crash. It **quietly sends you somewhere nearby**. That quiet guess is the danger.

Our pipeline uses `OneHotEncoder(handle_unknown="ignore")`. That means: **an unknown category becomes all zeros — no error, no warning.** Let's find unseen values ourselves.

In [ ]:
# 🔧 YOUR TURN: which category values appear in production but NEVER in reference?
for col in CAT:
    new_values = set(production[col]) ____ set(reference[col])
    print(f"{col:<11} -> {sorted(new_values) if new_values else 'nothing new'}")

<details><summary>▸ Hint</summary>

Python set difference: `set_a - set_b` gives the items in A that are not in B.
</details>

In [ ]:
three_gen = production[production.flat_model == "3Gen"]
print(f"First 3Gen resale: {three_gen.month.min()}, total so far: {len(three_gen)} deals")
print("Model v1 on 3Gen flats:", {k: round(v) for k, v in score(three_gen[TARGET], model_v1.predict(three_gen[FEATURES])).items()})

Only a few dozen 3Gen deals, so it barely changes the overall score — **which is exactly why nobody notices**. A buyer of a 3Gen flat, though, gets a poor estimate.

### Simulate a brand-new town (what will happen when Tengah arrives)
We pretend a town does not exist: train a model **without** it, then ask the model to price flats there.
We compare with a model that **did** see the town. We try two very different towns:
* **Punggol** — a young estate, similar to Sengkang next door.
* **Queenstown** — a mature, central estate with its own price premium.

In [ ]:
train_years = df[df.date < "2021-01-01"]                  # train on 2017–2020
test_2021   = df[df.date.dt.year == 2021]                  # test on 2021
model_all_towns = make_model().fit(train_years[FEATURES], train_years[TARGET])

for town in ["PUNGGOL", "QUEENSTOWN"]:
    # 🔧 YOUR TURN: keep every town EXCEPT this one
    train_without = train_years[train_years.town ____ town]
    model_without = make_model().fit(train_without[FEATURES], train_without[TARGET])

    rows = test_2021[test_2021.town == town]
    seen   = score(rows[TARGET], model_all_towns.predict(rows[FEATURES]))
    unseen = score(rows[TARGET], model_without.predict(rows[FEATURES]))
    print(f"{town:<11} town SEEN in training: MAPE {seen['MAPE_%']:5.1f}%  |  "
          f"NEVER seen: MAPE {unseen['MAPE_%']:5.1f}%, bias S${unseen['bias']:>9,.0f}")

<details><summary>▸ Hint</summary>

"Not equal to" in pandas is `!=`.
</details>

**What do you notice?**
* **Punggol:** hardly any difference. A young estate looks like other young estates, so the model's guess is close.
* **Queenstown:** the error jumps about **4–5 times**, and the model badly **under-prices** it. Its location premium is something the model could only learn from Queenstown's own sales.
* In both cases: **no error message.**

**So for real life:** Tengah (a young, non-mature town) will probably behave like Punggol. A new estate in a **prime, central location** will behave like Queenstown — exactly the kind of flat the new **Prime** classification is for.

### A simple guard you can add to any project

In [ ]:
def check_new_batch(batch, reference, cat_cols, expected_columns):
    # Run on every new batch of data BEFORE predicting.
    warnings_found = []
    missing = set(expected_columns) - set(batch.columns)
    extra   = set(batch.columns) - set(expected_columns)
    if missing: warnings_found.append(f"Missing columns: {sorted(missing)}")
    if extra:   warnings_found.append(f"New columns (schema change?): {sorted(extra)}")
    for col in cat_cols:
        unseen = set(batch[col].dropna()) - set(reference[col].dropna())
        if unseen:
            n = batch[col].isin(unseen).sum()
            warnings_found.append(f"{col}: {n} rows with unseen values {sorted(unseen)}")
    return warnings_found or ["✅ No new categories or schema changes"]


june_2022 = production[production.month == "2022-06"]
for w in check_new_batch(june_2022, reference, CAT, reference.columns):
    print(w)

💡 When HDB adds the **Standard / Plus / Prime** label to resale data, it will most likely arrive as a **new column** → the `extra` check above catches it.

**What to do when the guard fires:** (1) flag those predictions as *low confidence* in the app, (2) map the new value to the closest known one *only if a human agrees*, (3) collect enough rows, then retrain.

---
## Part F · 🔧 YOUR TURN — run drift checks on *your group's* model

Use the functions above on your own project data. Pick a cut-off date: everything **before** = reference, everything **after** = production.

In [ ]:
# my_df = ...                                # your group's cleaned dataset (needs a date column)
# CUTOFF = "2023-01-01"                      # pretend your model went live here
# my_ref, my_new = my_df[my_df.date < CUTOFF], my_df[my_df.date >= CUTOFF]
#
# for col in ["your", "numeric", "columns"]:
#     v = psi(my_ref[col], my_new[col]); print(f"{col:<25} PSI = {v:.3f} {traffic_light(v)}")
# for col in ["your", "category", "columns"]:
#     v = psi_categorical(my_ref[col], my_new[col]); print(f"{col:<25} PSI = {v:.3f} {traffic_light(v)}")
# for w in check_new_batch(my_new, my_ref, ["your", "category", "columns"], my_ref.columns): print(w)

---
## ✍️ Reflect (2 minutes, with your group)
1. Which of your features drifted the most? Is it a **bug** or the **real world**?
2. Your remaining-lease column drifts every year *by design*. Would you fire an alarm for it every month? What would you do instead?
3. All inputs except lease looked 🟢. Does that prove the model is still accurate? *(Notebook 02 answers this.)*

### Key takeaways
* **Data drift** = the inputs look different from training. You can check it **without** waiting for real prices.
* **PSI** = one number per column. Rule of thumb: < 0.1 🟢, 0.1–0.25 🟡, > 0.25 🔴.
* **KS / chi-square p-values** ring for tiny changes on big data — decide with **effect size** (PSI, D).
* **New categories** fail *silently*. Check for them on every batch.